# Modelo Elo-Poisson Dinâmico

Implementação do modelo de *Stochastic Gradient Ascent* sobre a log-verossimilhança de Poisson, conforme derivado no documento teórico. As taxas esperadas de gols são:

$$\lambda_A = \exp(\alpha_A - \beta_B), \quad \lambda_B = \exp(\alpha_B - \beta_A)$$

A regra de atualização após cada partida com placar $(x, y)$ é um passo de SGD **ponderado** pelo peso $w$ da competição:

$$\alpha_A \mathrel{+}= \eta \cdot w \cdot (x - \lambda_A), \quad \beta_A \mathrel{-}= \eta \cdot w \cdot (y - \lambda_B)$$
$$\alpha_B \mathrel{+}= \eta \cdot w \cdot (y - \lambda_B), \quad \beta_B \mathrel{-}= \eta \cdot w \cdot (x - \lambda_A)$$

Os pesos $w$ seguem a mesma escala de importância usada nos modelos Bayesianos do projeto (Copa do Mundo = 6, amistoso = 1). Todos os parâmetros iniciam em zero — prior neutro de 1 gol esperado por time. O período de treinamento é controlado pela variável `START_DATE`.

In [1]:
import sys

sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import poisson as sp_poisson

from src.constants import GROUPS
from src.data import prepare_cycle_data

/home/michels/WorldCup2026/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Carregamento e preparação dos dados

`START_DATE` define o início do período de treinamento. A função `prepare_cycle_data` aplica o mesmo filtro de volume mínimo de partidas usado nos modelos Bayesianos e calcula a coluna `game_weight` — peso de cada partida baseado na importância da competição. A tabela de pesos por torneio é exibida na célula seguinte.

In [2]:
START_DATE = "2019-01-01"

df, teams, team_map = prepare_cycle_data(
    "../data/raw/results.csv",
    start_date=START_DATE,
    apply_decay=False,
)

df = df.sort_values("date").reset_index(drop=True)

print(f"Período: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Partidas: {len(df):,}  |  Seleções: {len(teams)}")
df[
    [
        "date",
        "home_team",
        "away_team",
        "home_score",
        "away_score",
        "tournament",
        "game_weight",
    ]
].head(10)

Dados preparados: 6809 jogos e 211 seleções identificadas.
Período: 2019-01-02 → 2026-03-31
Partidas: 6,809  |  Seleções: 211


,date,home_team,away_team,home_score,away_score,tournament,game_weight
0,2019-01-02,Oman,Thailand,2,0,Friendly,1.0
1,2019-01-05,United Arab Emirates,Bahrain,1,1,AFC Asian Cup,4.0
2,2019-01-06,Thailand,India,1,4,AFC Asian Cup,4.0
3,2019-01-06,Australia,Jordan,0,1,AFC Asian Cup,4.0
4,2019-01-06,Syria,Palestine,0,0,AFC Asian Cup,4.0
5,2019-01-07,China PR,Kyrgyzstan,2,1,AFC Asian Cup,4.0
6,2019-01-07,South Korea,Philippines,1,0,AFC Asian Cup,4.0
7,2019-01-07,Iran,Yemen,5,0,AFC Asian Cup,4.0
8,2019-01-08,Finland,Sweden,1,0,Friendly,1.0
9,2019-01-08,Iraq,Vietnam,3,2,AFC Asian Cup,4.0


In [3]:
# Pesos por competição — mesma lógica de get_importance_factor
# usada em src/data/loader.py
weight_table = (
    df.groupby("tournament")["game_weight"]
    .first()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={"tournament": "Competição", "game_weight": "Peso (w)"})
)
weight_table

,Competição,Peso (w)
0,FIFA World Cup,6.0
1,African Cup of Nations,4.0
2,AFC Asian Cup,4.0
3,Copa América,4.0
4,UEFA Euro,4.0
5,Oceania Nations Cup,4.0
6,Gold Cup,4.0
7,EAFF Championship qualification,2.5
8,AFF Championship qualification,2.5
9,Arab Cup qualification,2.5


## 2. Implementação do modelo

O método `update` recebe o peso `w` da partida e escala o passo de gradiente por ele: competições mais importantes corrigem os parâmetros com maior intensidade. O método `fit` lê a coluna `game_weight` do DataFrame gerado por `prepare_cycle_data` e a repassa automaticamente a cada atualização.

In [4]:
class EloPoissonModel:
    """Online Elo-Poisson model via weighted SGD on the Poisson log-likelihood."""

    def __init__(self, teams: list[str], eta: float = 0.05):
        self.teams = teams
        self.eta = eta
        self.alpha = {t: 0.0 for t in teams}
        self.beta = {t: 0.0 for t in teams}

    def expected_goals(self, home: str, away: str) -> tuple[float, float]:
        lam_home = np.exp(self.alpha[home] - self.beta[away])
        lam_away = np.exp(self.alpha[away] - self.beta[home])
        return lam_home, lam_away

    def update(self, home: str, away: str, x: int, y: int, weight: float = 1.0) -> None:
        """One weighted SGD step after observing score (x, y)."""
        lam_a, lam_b = self.expected_goals(home, away)
        step = self.eta * weight
        self.alpha[home] += step * (x - lam_a)
        self.beta[home] -= step * (y - lam_b)
        self.alpha[away] += step * (y - lam_b)
        self.beta[away] -= step * (x - lam_a)

    def fit(self, df) -> "EloPoissonModel":
        """Train in chronological order, weighting each match by game_weight."""
        has_weight = "game_weight" in df.columns
        for row in df.itertuples(index=False):
            home, away = row.home_team, row.away_team
            if home in self.alpha and away in self.alpha:
                w = row.game_weight if has_weight else 1.0
                self.update(
                    home, away, int(row.home_score), int(row.away_score), weight=w
                )
        return self

    def parameters(self):
        return pd.DataFrame(
            {
                "team": self.teams,
                "alpha": [self.alpha[t] for t in self.teams],
                "beta": [self.beta[t] for t in self.teams],
            }
        ).set_index("team")

## 3. Treino do modelo

O modelo é treinado em ordem cronológica sobre todas as partidas do período. Cada jogo atualiza os quatro parâmetros das seleções envolvidas com intensidade proporcional ao peso da competição.

In [5]:
ETA = 0.05

model = EloPoissonModel(teams, eta=ETA).fit(df)

params = model.parameters()
params["strength"] = params["alpha"] + params["beta"]
params = params.sort_values("strength", ascending=False)

print(f"Modelo treinado com η = {ETA}")
print(f"Partidas processadas: {len(df):,}")
params.head(20)

Modelo treinado com η = 0.05
Partidas processadas: 6,809


,alpha,beta,strength
team,,,
Morocco,1.255726,1.571597,2.827323
Argentina,1.223104,1.386187,2.609291
England,1.065987,1.177002,2.242989
Japan,1.247890,0.992375,2.240265
Australia,1.209647,0.928865,2.138512
Germany,1.282323,0.841814,2.124137
Colombia,1.424105,0.635886,2.059991
Ecuador,0.497032,1.537691,2.034723
Brazil,1.156273,0.820377,1.976649


## 4. Análise dos parâmetros

Scatter de força de ataque (α) versus solidez defensiva (β) para todas as seleções treinadas. As 48 classificadas para a Copa 2026 são destacadas em vermelho.

In [6]:
cup_teams = {t for grp in GROUPS.values() for t in grp}
others = params[~params.index.isin(cup_teams)]
cup = params[params.index.isin(cup_teams)]

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=others["alpha"],
        y=others["beta"],
        mode="markers",
        marker={"color": "royalblue", "size": 7, "opacity": 0.5},
        name="Demais seleções",
    )
)
fig.add_trace(
    go.Scatter(
        x=cup["alpha"],
        y=cup["beta"],
        mode="markers+text",
        marker={"color": "crimson", "size": 9},
        text=cup.index,
        textposition="top center",
        textfont={"size": 8},
        name="Classificadas Copa 2026",
    )
)
fig.add_hline(y=0, line_width=0.8, line_dash="dash", line_color="gray")
fig.add_vline(x=0, line_width=0.8, line_dash="dash", line_color="gray")
fig.update_layout(
    title=(
        "Ataque × Defesa — Modelo Elo-Poisson"
        "<br><sup>vermelho = classificadas para Copa 2026</sup>"
    ),
    xaxis_title="Força de Ataque (α)",
    yaxis_title="Solidez Defensiva (β)",
    height=600,
    plot_bgcolor="white",
    paper_bgcolor="white",
)
fig.update_xaxes(showgrid=True, gridcolor="lightgrey", zeroline=False)
fig.update_yaxes(showgrid=True, gridcolor="lightgrey", zeroline=False)
fig.show()

## 5. Predição e probabilidades para partidas específicas

Com os parâmetros finais, calculamos as taxas esperadas de gols $\lambda_A$ e $\lambda_B$ para qualquer confronto e derivamos a distribuição conjunta de placares via produto de Poisson. As probabilidades de vitória, empate e derrota são obtidas somando as regiões correspondentes da matriz de placares.

In [7]:
MAX_GOALS = 8


def match_probs(model, home, away):
    lam_h, lam_a = model.expected_goals(home, away)
    goals = np.arange(0, MAX_GOALS + 1)
    mat = np.outer(sp_poisson.pmf(goals, lam_h), sp_poisson.pmf(goals, lam_a))
    return {
        "home_team": home,
        "away_team": away,
        "λ_home": round(lam_h, 3),
        "λ_away": round(lam_a, 3),
        "P(home win)": round(float(np.tril(mat, -1).sum()), 3),
        "P(draw)": round(float(np.trace(mat)), 3),
        "P(away win)": round(float(np.triu(mat, 1).sum()), 3),
    }


matchups = [
    ("Brazil", "Morocco"),
    ("Argentina", "Austria"),
    ("France", "Senegal"),
    ("Spain", "Uruguay"),
]

rows = [
    match_probs(model, h, a)
    for h, a in matchups
    if h in model.alpha and a in model.alpha
]
pd.DataFrame(rows)

,home_team,away_team,λ_home,λ_away,P(home win),P(draw),P(away win)
0,Brazil,Morocco,0.660,1.546,0.158,0.255,0.587
1,Argentina,Austria,2.085,0.446,0.760,0.173,0.067
2,France,Senegal,1.369,0.985,0.457,0.273,0.270
3,Spain,Uruguay,1.015,0.670,0.429,0.335,0.236


In [8]:
valid = [(h, a) for h, a in matchups if h in model.alpha and a in model.alpha]
goals_labels = list(range(MAX_GOALS + 1))

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[f"{h} × {a}" for h, a in valid],
    vertical_spacing=0.12,
    horizontal_spacing=0.08,
)

for (home, away), (r, c) in zip(valid, [(1, 1), (1, 2), (2, 1), (2, 2)], strict=False):
    lam_h, lam_a = model.expected_goals(home, away)
    goals = np.arange(0, MAX_GOALS + 1)
    M = np.outer(sp_poisson.pmf(goals, lam_h), sp_poisson.pmf(goals, lam_a)) * 100
    text = [[f"{M[i,j]:.1f}%" for j in goals_labels] for i in goals_labels]
    fig.add_trace(
        go.Heatmap(
            z=M,
            x=goals_labels,
            y=goals_labels,
            text=text,
            texttemplate="%{text}",
            textfont={"size": 8},
            colorscale="YlOrRd",
            showscale=False,
            xgap=1,
            ygap=1,
        ),
        row=r,
        col=c,
    )
    fig.update_xaxes(title_text=f"Gols {away}", row=r, col=c, tickvals=goals_labels)
    fig.update_yaxes(title_text=f"Gols {home}", row=r, col=c, tickvals=goals_labels)

fig.update_layout(
    title_text="Distribuição de Placares — Modelo Elo-Poisson",
    height=700,
    plot_bgcolor="white",
    paper_bgcolor="white",
)
fig.show()

## 6. Evolução temporal dos parâmetros

Reprocessamos o dataset em ordem cronológica com um modelo separado para registrar o estado de α, β e α+β de cada seleção antes de cada atualização. Os pesos por competição são usados também nesse passo, garantindo consistência com o modelo treinado na seção 3.

In [9]:
TRACK_TEAMS = [
    "Brazil",
    "Argentina",
    "France",
    "England",
    "Germany",
    "Spain",
    "Morocco",
    "Colombia",
]
TRACK_TEAMS = [t for t in TRACK_TEAMS if t in teams]

history = {
    t: {"alpha": [], "beta": [], "strength": [], "date": []} for t in TRACK_TEAMS
}
tracker = EloPoissonModel(teams, eta=ETA)

has_weight = "game_weight" in df.columns
for row in df.itertuples(index=False):
    home, away = row.home_team, row.away_team
    if home not in tracker.alpha or away not in tracker.alpha:
        continue
    for t in (home, away):
        if t in history:
            a = tracker.alpha[t]
            b = tracker.beta[t]
            history[t]["alpha"].append(a)
            history[t]["beta"].append(b)
            history[t]["strength"].append(a + b)
            history[t]["date"].append(row.date)
    w = row.game_weight if has_weight else 1.0
    tracker.update(home, away, int(row.home_score), int(row.away_score), weight=w)


colors = [
    "#1f77b4",
    "#ff7f0e",
    "#2ca02c",
    "#d62728",
    "#9467bd",
    "#8c564b",
    "#e377c2",
    "#bcbd22",
]
fig = make_subplots(
    rows=3,
    cols=1,
    subplot_titles=["Ataque (α)", "Defesa (β)", "Força composta (α + β)"],
    vertical_spacing=0.08,
)
for i, t in enumerate(TRACK_TEAMS):
    h = history[t]
    if not h["date"]:
        continue
    color = colors[i % len(colors)]
    common = {"name": t, "line": {"color": color}, "legendgroup": t}
    fig.add_trace(
        go.Scatter(x=h["date"], y=h["alpha"], mode="lines", **common), row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=h["date"], y=h["beta"], mode="lines", **common, showlegend=False),
        row=2,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=h["date"], y=h["strength"], mode="lines", **common, showlegend=False
        ),
        row=3,
        col=1,
    )

for r in (1, 2, 3):
    fig.add_hline(
        y=0, line_width=0.8, line_dash="dash", line_color="gray", row=r, col=1
    )

fig.update_yaxes(
    title_text="α", row=1, col=1, showgrid=True, gridcolor="lightgrey", zeroline=False
)
fig.update_yaxes(
    title_text="β", row=2, col=1, showgrid=True, gridcolor="lightgrey", zeroline=False
)
fig.update_yaxes(
    title_text="α + β",
    row=3,
    col=1,
    showgrid=True,
    gridcolor="lightgrey",
    zeroline=False,
)
fig.update_xaxes(showgrid=True, gridcolor="lightgrey", zeroline=False)
fig.update_layout(
    title_text="Trajetória dos parâmetros — principais seleções",
    height=900,
    plot_bgcolor="white",
    paper_bgcolor="white",
)
fig.show()

## 7. Ranking final das seleções classificadas para a Copa 2026

Ordenação das 48 seleções pela força composta α + β, que combina capacidade ofensiva e solidez defensiva aprendidas pelo modelo ao longo do período de treinamento.

In [10]:
group_of = {t: g for g, ts in GROUPS.items() for t in ts}
cup_teams = {t for grp in GROUPS.values() for t in grp}

ranking = (
    params.loc[params.index.isin(cup_teams)]
    .copy()
    .sort_values("strength", ascending=False)
    .reset_index()
)
ranking.insert(0, "pos", range(1, len(ranking) + 1))
ranking["group"] = ranking["team"].map(group_of)
ranking[["alpha", "beta", "strength"]] = ranking[["alpha", "beta", "strength"]].round(4)

print(f"Ranking Elo-Poisson — {len(ranking)} seleções classificadas")
ranking[["pos", "team", "group", "alpha", "beta", "strength"]]

Ranking Elo-Poisson — 48 seleções classificadas


,pos,team,group,alpha,beta,strength
0,1,Morocco,C,1.2557,1.5716,2.8273
1,2,Argentina,J,1.2231,1.3862,2.6093
2,3,England,L,1.0660,1.1770,2.2430
3,4,Japan,F,1.2479,0.9924,2.2403
4,5,Australia,D,1.2096,0.9289,2.1385
5,6,Germany,E,1.2823,0.8418,2.1241
6,7,Colombia,K,1.4241,0.6359,2.0600
7,8,Ecuador,E,0.4970,1.5377,2.0347
8,9,Brazil,C,1.1563,0.8204,1.9766
9,10,Spain,H,1.0781,0.8588,1.9369
